> **Notebook-first lesson.** Run cells in order. The final activity is designed to be changed and rerun.

## Mathematical Framework

Math companions for this lesson:

- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Track **shapes, assumptions, objective, derivatives/updates, and the conditions under which the derivation stops matching reality**.

# Lesson 27: Dataset and DataLoader

Real ML systems rarely fit all training data into one tensor operation.

## Dataset


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEED)
X_train = torch.randn(160, 2)
y_train = (X_train[:, 0] + .5 * X_train[:, 1] > 0).long()
X_val = torch.randn(60, 2)
y_val = (X_val[:, 0] + .5 * X_val[:, 1] > 0).long()
device = torch.device('cpu')
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.01)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)


In [ ]:
from torch.utils.data import Dataset

class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]



## DataLoader


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    SimpleDataset(X_train, y_train),
    batch_size=64,
    shuffle=True,
)



## Training by mini-batch


In [ ]:
for xb, yb in loader:
    logits = model(xb)
    loss = loss_fn(logits, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



## Concepts
Batch size affects memory use, gradient noise and training throughput.

Shuffling breaks accidental ordering patterns.

Custom Datasets can load files lazily rather than loading everything into memory.

## Exercise
Create a Dataset that stores synthetic signal windows and labels. Return:
- waveform tensor
- signal class
- SNR metadata

Then build a DataLoader and inspect one batch.


## Runnable activity
Run this experiment. Then change one architectural, data, or optimization choice and compare.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
waveforms=torch.randn(128,256)
labels=torch.randint(0,4,(128,))
snr=torch.empty(128).uniform_(-10,20)
ds=TensorDataset(waveforms,labels,snr)
loader=DataLoader(ds,batch_size=16,shuffle=True)
xb,yb,sb=next(iter(loader))
print("waveforms",xb.shape,"labels",yb.shape,"snr",sb.shape)
print("batch SNR range",float(sb.min()),float(sb.max()))

## Explanation checkpoint
Add a Markdown cell that explains the tensor shapes, the mechanism being tested, and what changed when you modified the experiment.